In [1]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — Setup
# ═══════════════════════════════════════════════════════════════════
import subprocess, sys, os, time, warnings, re as _re, threading
from itertools import cycle
warnings.filterwarnings('ignore')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'casanovo>=5.0.0', 'pyteomics', 'lxml', 'remotezip', 'appdirs'], check=True)

import numpy as np, pandas as pd, datetime
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from pathlib import Path

torch.manual_seed(42)
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_NAME   = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'
TOTAL_VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == 'cuda' else 0
N_PEAKS    = 150

BATCH_SIZES      = [1, 8, 32, 128, 512]
N_SUBSET_SWEEP   = 5000   # real spectra in MGF; large enough that cycling is minimal
N_TIMING_SWEEP   = 5000   # target timed spectra per variant per batch size
N_WARMUP         = 3
N_COMPILE_WARMUP = 8

LABEL_BASE    = 'Baseline'
LABEL_VEC     = 'Vec only (P1-4)'
LABEL_COMPILE = 'Compile only (P5)'
LABEL_OPT     = 'All patches (P1-5)'
ENABLE_COMPILE = True

print(f'Device: {DEVICE} | GPU: {GPU_NAME} | VRAM: {TOTAL_VRAM:.1f} GB')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')
print(f'Batch sizes: {BATCH_SIZES} | N_SUBSET_SWEEP={N_SUBSET_SWEEP} | N_TIMING_SWEEP={N_TIMING_SWEEP}')
os.makedirs('results', exist_ok=True)


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


Device: cuda | GPU: NVIDIA L4 | VRAM: 23.6 GB
PyTorch: 2.7.1+cu128 | CUDA: 12.8
Batch sizes: [1, 8, 32, 128, 512] | N_SUBSET_SWEEP=5000 | N_TIMING_SWEEP=5000


In [2]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — Download MGF
# ═══════════════════════════════════════════════════════════════════
from remotezip import RemoteZip
import shutil

ZIP_URL  = 'https://zenodo.org/records/12587317/files/mgf_data.zip?download=1'
TARGET   = 'multi-enzyme-simple.test.mgf'
MGF_PATH = TARGET

if not (os.path.exists(MGF_PATH) and os.path.getsize(MGF_PATH) > 1e6):
    with RemoteZip(ZIP_URL) as zf:
        src = next(n for n in zf.namelist() if TARGET in n)
        zf.extract(src, '.')
    if src != MGF_PATH and os.path.exists(src):
        shutil.move(src, MGF_PATH)
        top = src.split('/')[0]
        if os.path.isdir(top): shutil.rmtree(top, ignore_errors=True)

print(f'{MGF_PATH}  ({os.path.getsize(MGF_PATH)/1e6:.1f} MB)')

multi-enzyme-simple.test.mgf  (300.9 MB)


In [3]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — EDA (condensed)
# ═══════════════════════════════════════════════════════════════════
def parse_mgf(path):
    records, spec, peaks, in_s = [], {}, [], False
    with open(path, 'r', errors='replace') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            if line.upper() == 'BEGIN IONS':   spec, peaks, in_s = {}, [], True
            elif line.upper() == 'END IONS':
                if in_s: records.append({'pepmass': spec.get('_pm', 0.),
                                          'charge':  spec.get('_ch', 1),
                                          'n_peaks': len(peaks)})
                in_s = False
            elif in_s:
                if '=' in line:
                    k, _, v = line.partition('='); k = k.strip().upper()
                    if k == 'PEPMASS': spec['_pm'] = float(v.strip().split()[0])
                    elif k == 'CHARGE': spec['_ch'] = int(_re.sub(r'[^\d]', '', v.strip()) or '1')
                else:
                    try: peaks.append(float(line.split()[0]))
                    except: pass
    return pd.DataFrame(records)

eda = parse_mgf(MGF_PATH)
print(f'Dataset: {len(eda):,} spectra | charges +{eda.charge.min()}-+{eda.charge.max()} '
      f'| avg peaks: {eda.n_peaks.mean():.0f}')

Dataset: 106,933 spectra | charges +1-+8 | avg peaks: 123


In [4]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — Load model + create 4 variants via runtime monkey-patching
# ═══════════════════════════════════════════════════════════════════
import copy, appdirs, inspect, types, re, textwrap
from casanovo.casanovo import _get_model_weights
from casanovo.denovo.model_runner import ModelRunner
from casanovo.denovo import model as _casanovo_model_mod
from casanovo.config import Config

cache_dir = Path(appdirs.user_cache_dir("casanovo", False, opinion=False))
ckpt_path = _get_model_weights(cache_dir)
print(f'Checkpoint: {ckpt_path}  ({os.path.getsize(str(ckpt_path))/1e6:.0f} MB)')

config = Config()
runner = ModelRunner(config=config, model_filename=str(ckpt_path))
runner.initialize_tokenizer()
runner.initialize_model(train=False)
model_base = runner.model.to(DEVICE).eval()
print(f'Baseline: {type(model_base).__name__} | '
      f'{sum(p.numel() for p in model_base.parameters())/1e6:.1f}M params')

_mod_globals = vars(_casanovo_model_mod)

# ── Generic patch helper (for single-method replacements) ─────────
def patch_method(instance, method_name, replacements):
    src = textwrap.dedent(inspect.getsource(getattr(type(instance), method_name)))
    for pat, repl, flags, tag in replacements:
        src, n = re.subn(pat, repl, src, flags=flags)
        print(f'    {"ok" if n else "WARN"} {tag} ({n} sub)')
    try:
        locs = {}
        exec(compile(src, f'<{method_name}>', 'exec'), _mod_globals, locs)
        setattr(instance, method_name, types.MethodType(locs[method_name], instance))
    except Exception as e:
        print(f'    ERR bind: {e}')

# ── Patch 4: module-level _peptide_score (applied once) ──────────
def _patched_peptide_score(aa_scores, fits_precursor_mz=True, lengths=None):
    import numpy as _np
    if isinstance(aa_scores, torch.Tensor):
        eps = torch.finfo(torch.float64).eps
        if aa_scores.ndim == 1:
            s = torch.exp(torch.sum(torch.log(torch.clamp(aa_scores.double(), eps, 1))))
            return s - 1 if not fits_precursor_mz else s
        if lengths is None: raise ValueError('lengths required')
        cs = torch.cumsum(torch.log(torch.clamp(aa_scores.double(), eps, 1)), dim=1)
        idx = torch.arange(aa_scores.shape[0], device=aa_scores.device)
        scores = torch.exp(cs[idx, torch.clamp(lengths - 1, min=0)])
        if isinstance(fits_precursor_mz, bool):
            if not fits_precursor_mz: scores = scores - 1
        else: scores[~fits_precursor_mz] -= 1
        return scores
    eps = _np.finfo(_np.float64).eps
    if aa_scores.ndim == 1:
        s = _np.exp(_np.sum(_np.log(_np.clip(aa_scores, eps, 1))))
        if not fits_precursor_mz: s -= 1
        return s
    if lengths is None: raise ValueError('lengths required')
    cs = _np.cumsum(_np.log(_np.clip(aa_scores, eps, 1)), axis=1)
    idx = _np.arange(aa_scores.shape[0])
    scores = _np.exp(cs[idx, _np.maximum(lengths - 1, 0)])
    if isinstance(fits_precursor_mz, (bool, _np.bool_)):
        if not fits_precursor_mz: scores -= 1
    else: scores[~fits_precursor_mz] -= 1
    return scores

_casanovo_model_mod._peptide_score = _patched_peptide_score
_mod_globals['_peptide_score']     = _patched_peptide_score
print('Patch 4 (_peptide_score): ok')

# ── Patches 1-3 applied to instance ──────────────────────────────
def apply_vec_patches(m, lbl):
    print(f'\nPatches 1-3 -> [{lbl}]')

    # Patch 1: cumulative_masses nested loop -> single tensor gather
    patch_method(m, 'beam_search_decode', [(
        (r'cumulative_masses\s*=\s*torch\.zeros\(batch,\s*beam,\s*device=device\)\s+'
         r'for b in range\(batch\):\s+for s in range\(beam\):\s+'
         r'token_idx\s*=\s*tokens\[b,\s*0,\s*s\]\.item\(\)\s+'
         r'if token_idx\s*<\s*len\(token_masses\):\s*[^\n]*\n'
         r'\s+cumulative_masses\[b,\s*s\]\s*=\s*token_masses\[token_idx\]'),
        'cumulative_masses = token_masses[tokens[:, 0, :]]',
        re.DOTALL, 'P1: cumulative_masses vectorize'
    )])

    # Patch 2: _finish_beams per-beam loop -> single batched tokenizer call
    _src_fb = textwrap.dedent(inspect.getsource(getattr(type(m), '_finish_beams')))
    _mf     = re.search(r'^(\s+)for i, beam_idx in enumerate\(idx\)', _src_fb, re.MULTILINE)
    if _mf:
        _ind = _mf.group(1)
        if 'sequences_to_check_t' not in _src_fb:
            _setup = (f'# P2 inject\n{_ind}sequences_to_check_t = tokens[idx, : step + 1]\n'
                      f'{_ind}_sct_is_stop = sequences_to_check_t == self.stop_token\n'
                      f'{_ind}sequences_to_check_t = torch.where(_sct_is_stop, '
                      f'torch.zeros_like(sequences_to_check_t), sequences_to_check_t)\n{_ind}')
            _src_fb, _ = re.subn(
                r'(?m)^(' + re.escape(_ind) + r'for i, beam_idx in enumerate\(idx\):)',
                _ind + _setup + r'for i, beam_idx in enumerate(idx):', _src_fb, count=1)
        _vec = (f'# P2 batched\n{_ind}recalc_mzs = self.tokenizer.calculate_precursor_ions('
                f'\n{_ind}    sequences_to_check_t.cpu(), charges_to_check.cpu()'
                f'\n{_ind}).to(device, dtype=torch.float64)'
                f'\n{_ind}recalc_neutral_masses = (recalc_mzs - 1.007276) * charges_to_check.double()'
                f'\n{_ind}self._cumulative_masses[idx] = recalc_neutral_masses.to(self._cumulative_masses.dtype)'
                f'\n{_ind}current_mzs = recalc_mzs')
        _src_fb, n2 = re.subn(
            r'for i, beam_idx in enumerate\(idx\):.*?(?=\n[ \t]*precursor_mzs_obs\s*=)',
            _vec, _src_fb, count=1, flags=re.DOTALL)
        print(f'    {"ok" if n2 else "WARN"} P2: _finish_beams vectorize ({n2} sub)')
        try:
            locs = {}
            exec(compile(_src_fb, '<_finish_beams>', 'exec'), _mod_globals, locs)
            setattr(m, '_finish_beams', types.MethodType(locs['_finish_beams'], m))
        except Exception as e:
            print(f'    ERR _finish_beams: {e}')
    else:
        print('    WARN P2: for-loop anchor not found')

    # Patch 3: _cache_finished_beams — SINGLE comprehensive block replacement.
    #
    # ROOT CAUSE of previous NameError: 5 sequential small substitutions
    # (P3a..P3d) left aa_scores_tensor undefined in some execution paths
    # because each patch assumed the previous one had already run cleanly.
    # Fix: replace the ENTIRE score-extraction + scoring block in one shot,
    # which makes all variable names consistent and avoids ordering issues.
    _src_cf = textwrap.dedent(inspect.getsource(getattr(type(m), '_cache_finished_beams')))

    # Detect indentation from the range_tensor line (most stable anchor)
    _mi  = re.search(r'^(\s+)range_tensor\s*=\s*torch\.arange', _src_cf, re.MULTILINE)
    _ic  = _mi.group(1) if _mi else '            '
    _ic4 = _ic + '    '

    # Pattern: matches from `range_tensor = torch.arange(...)` through
    # `aa_scores = aa_scores[:-1]` (the entire score extraction block).
    # The `.*?` with DOTALL absorbs the comment, if-block, and _peptide_score call.
    _p3_pat = (
        r'range_tensor\s*=\s*torch\.arange\(len\(pred_tokens\),\s*device=device\)\s*\n'
        r'[ \t]*aa_scores\s*=\s*smx\[0,\s*range_tensor,\s*pred_tokens\]'
        r'\.cpu\(\)\.numpy\(\).*?'
        r'[ \t]*aa_scores\s*=\s*aa_scores\[:-1\]'
    )

    # Replacement: fully self-consistent block where aa_scores_tensor is
    # always defined before it is used, regardless of has_stop_token.
    _p3_repl = (
        f'range_tensor = torch.arange(len(pred_tokens), device=device)\n'
        f'{_ic}aa_scores_tensor = smx[0, range_tensor, pred_tokens]\n'
        f'{_ic}if not has_stop_token:\n'
        f'{_ic4}aa_scores_tensor = torch.cat([\n'
        f'{_ic4}    aa_scores_tensor,\n'
        f'{_ic4}    torch.tensor([0.0], device=aa_scores_tensor.device),\n'
        f'{_ic4}])\n'
        f'{_ic}peptide_score_val = _peptide_score(\n'
        f'{_ic4}aa_scores_tensor, beam_fits_precursor[i]\n'
        f'{_ic})\n'
        f'{_ic}if isinstance(peptide_score_val, torch.Tensor):\n'
        f'{_ic4}peptide_score_val = peptide_score_val.item()\n'
        f'{_ic}aa_scores = aa_scores_tensor.cpu().numpy()[:-1]'
    )

    _src_cf, n3 = re.subn(_p3_pat, _p3_repl, _src_cf, count=1, flags=re.DOTALL)
    print(f'    {"ok" if n3 else "WARN"} P3: score block replacement ({n3} sub)')

    # P3e: rename peptide_score -> peptide_score_val in heapadd tuple.
    # Must run AFTER P3 so the heapadd still has the original name to match.
    _src_cf, ne = re.subn(
        r'\bpeptide_score,(\s*\n\s*np\.random\.random_sample)',
        r'peptide_score_val,\1', _src_cf
    )
    print(f'    {"ok" if ne else "WARN"} P3e: heapadd rename ({ne} sub)')

    try:
        locs = {}
        exec(compile(_src_cf, '<_cache_finished_beams>', 'exec'), _mod_globals, locs)
        setattr(m, '_cache_finished_beams',
                types.MethodType(locs['_cache_finished_beams'], m))
        print(f'    -> _cache_finished_beams bound')
    except Exception as e:
        print(f'    ERR _cache_finished_beams: {e}')

    print(f'  Vec patches complete for [{lbl}]')


def apply_compile(m, lbl):
    if ENABLE_COMPILE:
        m.encoder = torch.compile(m.encoder, mode='default', fullgraph=False, dynamic=True)
        m.decoder = torch.compile(m.decoder, mode='default', fullgraph=False, dynamic=True)
        print(f'  torch.compile(dynamic=True) -> [{lbl}]')

# ── Build 3 non-baseline variants ─────────────────────────────────
print('\nCreating variants...')
model_vec     = copy.deepcopy(model_base).eval()
model_compile = copy.deepcopy(model_base).eval()
model_opt     = copy.deepcopy(model_base).eval()

apply_vec_patches(model_vec,  LABEL_VEC)
apply_compile(model_compile,  LABEL_COMPILE)
apply_vec_patches(model_opt,  LABEL_OPT)
apply_compile(model_opt,      LABEL_OPT)

# ── Sanity check: separate mz and intensity tensors ───────────────
# Using distinct tensors avoids edge-case issues with shared references.
_mz = torch.zeros(1, N_PEAKS, device=DEVICE)
_it = torch.zeros(1, N_PEAKS, device=DEVICE)
_mz[0, :10] = torch.rand(10, device=DEVICE) * 800 + 200
_it[0, :10] = torch.rand(10, device=DEVICE)
_it[0, :10] /= _it[0, :10].sum()   # intensities must sum to ~1
_p  = torch.tensor([[1190., 2., 596.]], device=DEVICE)

print('\nSanity checks:')
for _m, _l in [(model_base,    LABEL_BASE),
               (model_vec,     LABEL_VEC),
               (model_compile, LABEL_COMPILE),
               (model_opt,     LABEL_OPT)]:
    with torch.no_grad(): _m.beam_search_decode(_mz, _it, _p)
    print(f'  {_l}: OK')

Checkpoint directory not set in ModelRunner, no checkpoint files will be saved.
Configured residue(s) not in model alphabet: N[Deamidated], [Carbamyl]-, [Ammonia-loss]-, C[Carbamidomethyl], [+25.980265]-, M[Oxidation], [Acetyl]-, Q[Deamidated]


Checkpoint: /home/zeus/.cache/casanovo/casanovo_v5_0_0_v5_0_0.ckpt  (575 MB)
Baseline: Spec2Pep | 47.9M params
Patch 4 (_peptide_score): ok

Creating variants...

Patches 1-3 -> [Vec only (P1-4)]
    ok P1: cumulative_masses vectorize (1 sub)
    ok P2: _finish_beams vectorize (1 sub)
    ok P3: score block replacement (1 sub)
    ok P3e: heapadd rename (1 sub)
    -> _cache_finished_beams bound
  Vec patches complete for [Vec only (P1-4)]
  torch.compile(dynamic=True) -> [Compile only (P5)]

Patches 1-3 -> [All patches (P1-5)]
    ok P1: cumulative_masses vectorize (1 sub)
    ok P2: _finish_beams vectorize (1 sub)
    ok P3: score block replacement (1 sub)
    ok P3e: heapadd rename (1 sub)
    -> _cache_finished_beams bound
  Vec patches complete for [All patches (P1-5)]
  torch.compile(dynamic=True) -> [All patches (P1-5)]

Sanity checks:
  Baseline: OK
  Vec only (P1-4): OK


W0605 15:10:08.201000 3791 /system/conda/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torch/_inductor/utils.py:1250] [0/0_1] Not enough SMs to use max_autotune_gemm mode
W0605 15:10:15.443000 3791 /system/conda/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torch/utils/_sympy/interp.py:176] [3/2_1] failed while executing pow_by_natural([VR[3, 4194303], VR[-1, -1]])


  Compile only (P5): OK
  All patches (P1-5): OK


In [5]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — Build sweep MGF + make_dm helper
# Rebuilds MGF and Lance DB automatically if N_SUBSET_SWEEP changes.
# ═══════════════════════════════════════════════════════════════════
from casanovo.denovo.dataloaders import DeNovoDataModule
import shutil as _shutil

SWEEP_MGF   = 'sweep_profile.mgf'
LANCE_SWEEP = os.path.join(os.getcwd(), 'lance_sweep')
os.makedirs(LANCE_SWEEP, exist_ok=True)

MODEL_MAX_CHARGE = model_base.decoder.charge_encoder.num_embeddings
print(f'Model max_charge: {MODEL_MAX_CHARGE}')

def write_subset_mgf(src, dest, n):
    count, buf, in_s = 0, [], False
    with open(src, 'r', errors='replace') as fin, open(dest, 'w') as fout:
        for line in fin:
            if count >= n: break
            if line.strip().upper() == 'BEGIN IONS':   in_s = True; buf = [line]
            elif line.strip().upper() == 'END IONS':
                buf.append(line); fout.writelines(buf); count += 1; in_s = False; buf = []
            elif in_s: buf.append(line)
    return count

# Markers: track both max_charge and N_SUBSET_SWEEP so any change triggers rebuild
_mc_marker  = os.path.join(LANCE_SWEEP, '.max_charge')
_ns_marker  = os.path.join(LANCE_SWEEP, '.n_subset')
_lance_test = os.path.join(LANCE_SWEEP, 'test.lance')
_prev_mc    = open(_mc_marker).read().strip() if os.path.exists(_mc_marker) else 'none'
_prev_ns    = open(_ns_marker).read().strip() if os.path.exists(_ns_marker) else 'none'

_mgf_stale   = (_prev_ns != str(N_SUBSET_SWEEP) or
                not os.path.exists(SWEEP_MGF) or
                os.path.getsize(SWEEP_MGF) < 1e5)
_lance_stale = (_prev_mc != str(MODEL_MAX_CHARGE) or _mgf_stale)

if _mgf_stale:
    n_wrote = write_subset_mgf(MGF_PATH, SWEEP_MGF, N_SUBSET_SWEEP)
    with open(_ns_marker, 'w') as f: f.write(str(N_SUBSET_SWEEP))
    print(f'Created {SWEEP_MGF}: {n_wrote} spectra')
else:
    print(f'Reusing {SWEEP_MGF}')

if _lance_stale:
    if os.path.exists(_lance_test): _shutil.rmtree(_lance_test)
    with open(_mc_marker, 'w') as f: f.write(str(MODEL_MAX_CHARGE))
    print(f'Rebuilding Lance DB (max_charge={MODEL_MAX_CHARGE}, n_subset={N_SUBSET_SWEEP})')
else:
    print(f'Reusing Lance DB')

def make_dm(bs):
    """Return DataModule for given batch size; reuses Lance DB across all batch sizes."""
    dm = DeNovoDataModule(
        lance_dir=LANCE_SWEEP,
        test_paths=[SWEEP_MGF],
        eval_batch_size=bs,
        tokenizer=runner.model.tokenizer,
        max_charge=MODEL_MAX_CHARGE,
        n_workers=0,
    )
    dm.setup(stage='test', annotated=False)
    return dm

_dm0 = make_dm(1)
_b0  = next(iter(_dm0.predict_dataloader()))
_mz0, _it0, _pr0, _ = model_base._process_batch(_b0)
n_lance_spectra = sum(1 for _ in _dm0.predict_dataloader())
print(f'Lance DB: {n_lance_spectra} spectra | batch shape: mzs={_mz0.shape}')
print(f'DataModule ready. N_TIMING_SWEEP={N_TIMING_SWEEP} | cycling ratio: '
      f'{N_TIMING_SWEEP / max(n_lance_spectra, 1):.1f}x')

Model max_charge: 4
Reusing sweep_profile.mgf
Reusing Lance DB


sweep_profile.mgf: 0 spectra [00:00, ? spectra/s]

Lance DB: 4485 spectra | batch shape: mzs=torch.Size([1, 42])
DataModule ready. N_TIMING_SWEEP=5000 | cycling ratio: 1.1x


In [6]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6 — Batch-size sweep: 4 variants × 5 batch sizes
#
# For each (bs, variant): cycles the sweep MGF to time N_TIMING_SWEEP
# spectra. Stops increasing bs per variant on OOM. Tracks GPU util
# and peak VRAM via background thread.
# ═══════════════════════════════════════════════════════════════════
from tqdm import tqdm

def _sync():
    if DEVICE == 'cuda': torch.cuda.synchronize()

class GPUMonitor:
    """Polls nvidia-smi every 0.3 s; falls back to torch VRAM if unavailable."""
    def __init__(self):
        self._util, self._vram, self._stop = [], [], threading.Event()
    def start(self):
        threading.Thread(target=self._run, daemon=True).start()
    def stop(self):
        self._stop.set()
    def _run(self):
        while not self._stop.is_set():
            try:
                r = subprocess.run(
                    ['nvidia-smi', '--query-gpu=utilization.gpu,memory.used',
                     '--format=csv,noheader,nounits'],
                    capture_output=True, text=True, timeout=1)
                if r.returncode == 0:
                    p = r.stdout.strip().split(', ')
                    self._util.append(float(p[0]))
                    self._vram.append(float(p[1]) / 1024)
            except Exception:
                if DEVICE == 'cuda':
                    self._vram.append(torch.cuda.memory_reserved() / 1e9)
            time.sleep(0.3)
    @property
    def avg_util(self): return float(np.mean(self._util)) if self._util else 0.
    @property
    def max_vram(self): return float(np.max(self._vram)) if self._vram else 0.

def time_at_bs(model, label, bs, n_timing, n_warmup):
    """
    Time model at given batch size over n_timing spectra (with cycling).
    Returns stats dict. Returns {'oom': True} on OutOfMemoryError.
    """
    try:
        loader = make_dm(bs).predict_dataloader()

        # Warmup: n_warmup batches (covers torch.compile JIT on first call)
        _cyc_w = cycle(loader)
        with torch.no_grad():
            for _ in range(n_warmup):
                b = next(_cyc_w)
                mz, it, pr, _ = model._process_batch(b)
                model.beam_search_decode(mz.to(DEVICE), it.to(DEVICE), pr.to(DEVICE))
        _sync()

        # Timed run
        mon = GPUMonitor(); mon.start()
        _cyc_t  = cycle(loader)
        per_spec_ms, n_done = [], 0

        while n_done < n_timing:
            b = next(_cyc_t)
            mz, it, pr, _ = model._process_batch(b)
            actual_bs = mz.shape[0]
            mz, it, pr = mz.to(DEVICE), it.to(DEVICE), pr.to(DEVICE)
            _sync(); t0 = time.perf_counter()
            with torch.no_grad():
                model.beam_search_decode(mz, it, pr)
            _sync()
            ms_batch = (time.perf_counter() - t0) * 1000
            # All spectra in a batch share the same wall-clock latency
            per_spec_ms.extend([ms_batch / actual_bs] * actual_bs)
            n_done += actual_bs

        mon.stop()
        mean_ms = float(np.mean(per_spec_ms))
        return dict(
            mean_ms   = round(mean_ms, 2),
            p50_ms    = round(float(np.percentile(per_spec_ms, 50)), 2),
            p95_ms    = round(float(np.percentile(per_spec_ms, 95)), 2),
            throughput= round(1000.0 / mean_ms, 2),
            gpu_pct   = round(mon.avg_util, 1),
            vram_gb   = round(mon.max_vram, 2),
            n_timed   = n_done,
            oom       = False,
        )
    except torch.cuda.OutOfMemoryError:
        if DEVICE == 'cuda': torch.cuda.empty_cache()
        return dict(oom=True, mean_ms=None, p50_ms=None, p95_ms=None,
                    throughput=None, gpu_pct=None, vram_gb=None, n_timed=0)

VARIANTS_SWEEP = [
    (model_base,    LABEL_BASE,    N_WARMUP,                    'base'),
    (model_vec,     LABEL_VEC,     N_WARMUP,                    'vec'),
    (model_compile, LABEL_COMPILE, N_WARMUP + N_COMPILE_WARMUP, 'compile'),
    (model_opt,     LABEL_OPT,     N_WARMUP + N_COMPILE_WARMUP, 'opt'),
]

# sweep_results[bs][variant_key] = stats dict
sweep_results = {}
oom_variants  = set()   # variants that OOM'd; skipped at all larger bs

for bs in BATCH_SIZES:
    print(f'\n{"="*55}')
    print(f'BATCH SIZE = {bs}  (target {N_TIMING_SWEEP} spectra each variant)')
    print(f'{"="*55}')
    sweep_results[bs] = {}

    for model, label, nwup, key in VARIANTS_SWEEP:
        if key in oom_variants:
            sweep_results[bs][key] = dict(oom=True, mean_ms=None, throughput=None,
                                          p50_ms=None, p95_ms=None,
                                          gpu_pct=None, vram_gb=None, n_timed=0)
            print(f'  {label:<35} skipped (OOM at smaller bs)')
            continue

        if DEVICE == 'cuda': torch.cuda.empty_cache()
        result = time_at_bs(model, label, bs, N_TIMING_SWEEP, nwup)
        sweep_results[bs][key] = result

        if result['oom']:
            oom_variants.add(key)
            print(f'  {label:<35} OOM')
        else:
            print(f'  {label:<35} '
                  f'{result["mean_ms"]:>8.1f} ms/spec | '
                  f'{result["throughput"]:>8.1f} spec/s | '
                  f'GPU {result["gpu_pct"]:>4.0f}% | '
                  f'VRAM {result["vram_gb"]:.2f} GB | '
                  f'n={result["n_timed"]}')

print('\nSweep complete.')


BATCH SIZE = 1  (target 5000 spectra each variant)


sweep_profile.mgf: 0 spectra [00:00, ? spectra/s]

  Baseline                               338.1 ms/spec |      3.0 spec/s | GPU   12% | VRAM 0.97 GB | n=5000


sweep_profile.mgf: 0 spectra [00:00, ? spectra/s]

  Vec only (P1-4)                        345.2 ms/spec |      2.9 spec/s | GPU   12% | VRAM 0.97 GB | n=5000


...

  Vec only (P1-4)                          9.1 ms/spec |    109.8 spec/s | GPU   86% | VRAM 2.97 GB | n=5509


sweep_profile.mgf: 0 spectra [00:00, ? spectra/s]

  Compile only (P5)                       10.2 ms/spec |     98.0 spec/s | GPU   75% | VRAM 2.75 GB | n=5509


sweep_profile.mgf: 0 spectra [00:00, ? spectra/s]

  All patches (P1-5)                       9.0 ms/spec |    110.7 spec/s | GPU   85% | VRAM 2.70 GB | n=5509

Sweep complete.


In [7]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7 — Batch-scaling comparison tables and speedup analysis
# ═══════════════════════════════════════════════════════════════════
VARIANT_KEYS = [('base', LABEL_BASE), ('vec', LABEL_VEC),
                ('compile', LABEL_COMPILE), ('opt', LABEL_OPT)]

# ── Table 1: full metrics per (bs, variant) ───────────────────────
flat_rows = []
for bs in BATCH_SIZES:
    for key, label in VARIANT_KEYS:
        r = sweep_results.get(bs, {}).get(key, {})
        if r.get('oom') or r.get('mean_ms') is None:
            flat_rows.append({'Batch Size': bs, 'Variant': label,
                               'Mean ms/spec': 'OOM', 'p50 ms': 'OOM',
                               'p95 ms': 'OOM', 'Throughput (spec/s)': 'OOM',
                               'GPU %': 'OOM', 'VRAM GB': 'OOM', 'n': 0})
        else:
            flat_rows.append({'Batch Size': bs, 'Variant': label,
                               'Mean ms/spec':        r['mean_ms'],
                               'p50 ms':              r['p50_ms'],
                               'p95 ms':              r['p95_ms'],
                               'Throughput (spec/s)': r['throughput'],
                               'GPU %':               r['gpu_pct'],
                               'VRAM GB':             r['vram_gb'],
                               'n':                   r['n_timed']})

df_sweep = pd.DataFrame(flat_rows)
print('BATCH-SCALING COMPARISON TABLE')
print('='*90)
print(df_sweep.to_string(index=False))
df_sweep.to_csv('results/batch_scaling_comparison.csv', index=False)

# ── Table 2: speedup of each optimized variant vs baseline ────────
speedup_rows = []
for bs in BATCH_SIZES:
    base_r = sweep_results.get(bs, {}).get('base', {})
    if base_r.get('oom') or base_r.get('throughput') is None: continue
    base_tp = base_r['throughput']
    row = {'Batch Size': bs, 'Baseline spec/s': base_tp}
    for key, label in VARIANT_KEYS[1:]:
        r = sweep_results.get(bs, {}).get(key, {})
        if r.get('oom') or r.get('throughput') is None:
            row[f'{label} spec/s'] = 'OOM'
            row[f'{label} speedup'] = 'OOM'
        else:
            row[f'{label} spec/s']  = r['throughput']
            row[f'{label} speedup'] = f'{r["throughput"] / base_tp:.3f}x'
    speedup_rows.append(row)

df_speedup = pd.DataFrame(speedup_rows)
print('\nSPEEDUP TABLE (vs Baseline)')
print('='*90)
print(df_speedup.to_string(index=False))
df_speedup.to_csv('results/batch_speedup_table.csv', index=False)

# ── Per-batch-size summary: best throughput ───────────────────────
print('\nPER-BATCH-SIZE BEST THROUGHPUT')
print('-'*60)
for bs in BATCH_SIZES:
    best_tp, best_lbl = 0., ''
    for key, label in VARIANT_KEYS:
        r = sweep_results.get(bs, {}).get(key, {})
        if not r.get('oom') and r.get('throughput') and r['throughput'] > best_tp:
            best_tp, best_lbl = r['throughput'], label
    if best_tp > 0:
        print(f'  bs={bs:<4}  best: {best_lbl:<35} {best_tp:.1f} spec/s')
    else:
        print(f'  bs={bs:<4}  all OOM')

# ── Numeric data for plotting (Cell 9) ───────────────────────────
plot_data = {}   # plot_data[key] = {'bs': [], 'tp': [], 'lat': [], 'gpu': [], 'vram': []}
for key, label in VARIANT_KEYS:
    plot_data[key] = {'bs': [], 'tp': [], 'lat': [], 'gpu': [], 'vram': [], 'label': label}
    for bs in BATCH_SIZES:
        r = sweep_results.get(bs, {}).get(key, {})
        if not r.get('oom') and r.get('throughput') is not None:
            plot_data[key]['bs'].append(bs)
            plot_data[key]['tp'].append(r['throughput'])
            plot_data[key]['lat'].append(r['mean_ms'])
            plot_data[key]['gpu'].append(r['gpu_pct'])
            plot_data[key]['vram'].append(r['vram_gb'])

print('\nSaved: results/batch_scaling_comparison.csv  batch_speedup_table.csv')

BATCH-SCALING COMPARISON TABLE
 Batch Size            Variant  Mean ms/spec  p50 ms  p95 ms  Throughput (spec/s)  GPU %  VRAM GB    n
          1           Baseline        338.13  315.47  567.06                 2.96   12.2     0.97 5000
          1    Vec only (P1-4)        345.16  319.80  578.29                 2.90   11.9     0.97 5000
          1  Compile only (P5)        343.71  319.64  574.44                 2.91   11.7     0.97 5000
          1 All patches (P1-5)        340.68  317.30  570.29                 2.94   11.8     0.97 5000
          8           Baseline         70.97   69.87   95.44                14.09   16.7     0.99 5005
          8    Vec only (P1-4)         68.98   67.75   92.49                14.50   17.0     0.99 5005
          8  Compile only (P5)         70.42   69.23   94.18                14.20   16.6     1.03 5005
          8 All patches (P1-5)         68.27   68.01   91.59                14.65   17.0     0.99 5005
         32           Baseline         23.

In [8]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8 — Micro-benchmarks: Patch 1, 2, 5 isolation
# Patch 3+4 (_peptide_score) removed: standalone op is not
# representative (kernel-launch overhead dominates for 14 elements).
# End-to-end batched timing in Cell 6 captures the real benefit.
# ═══════════════════════════════════════════════════════════════════
def bench(fn, n_reps=200, n_warmup=20):
    for _ in range(n_warmup): fn()
    if DEVICE == 'cuda': torch.cuda.synchronize()
    times = []
    for _ in range(n_reps):
        if DEVICE == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter(); fn()
        if DEVICE == 'cuda': torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times)), float(np.std(times))

micro_results = {}
micro_rows    = []

# ── MICRO 1 — Patch 1: cumulative_masses ─────────────────────────
print('='*60); print('MICRO 1: cumulative_masses (nested loop vs tensor gather)')
print('='*60)
BATCH, BEAM  = 4, 5
token_masses = model_base.token_masses.to(DEVICE)
vocab_size   = len(token_masses)
tokens_m1    = torch.randint(1, vocab_size - 1, (BATCH, BEAM), device=DEVICE)

def cm_loop():
    cm = torch.zeros(BATCH, BEAM, device=DEVICE, dtype=token_masses.dtype)
    for b in range(BATCH):
        for s in range(BEAM):
            iv = tokens_m1[b, s].item()
            if iv < vocab_size: cm[b, s] = token_masses[iv]
    return cm

def cm_vec(): return token_masses[tokens_m1]

assert torch.allclose(cm_loop(), cm_vec(), atol=1e-9), 'Equivalence FAILED'
print('  Equivalence: PASSED')
m1l, s1l = bench(cm_loop); m1v, s1v = bench(cm_vec)
_sp1 = m1l / m1v
print(f'  Loop:        {m1l:.4f} +/- {s1l:.4f} ms')
print(f'  Vectorized:  {m1v:.4f} +/- {s1v:.4f} ms')
print(f'  Speedup:     {_sp1:.1f}x  (eliminates batch*beam Python round-trips per decode init)')
micro_results['cm'] = {'loop': m1l, 'vec': m1v, 'speedup': _sp1}
micro_rows.append({'Optimization': 'P1: cumulative_masses',
                   'Before (ms)': round(m1l, 4), 'After (ms)': round(m1v, 4),
                   'Speedup_val': round(_sp1, 1), 'Speedup': f'{_sp1:.1f}x',
                   'Note': 'nested loop -> tensor gather; measured in isolation'})

# ── MICRO 2 — Patch 2: _finish_beams ─────────────────────────────
print('\n' + '='*60)
print('MICRO 2: _finish_beams (per-beam loop vs batched tokenizer call)')
print('='*60)
N_BC = 16; SEQ_LEN = 10
tok  = runner.model.tokenizer
s2   = torch.randint(1, vocab_size - 1, (N_BC, SEQ_LEN), device=DEVICE)
c2   = torch.randint(2, 4, (N_BC,), device=DEVICE)

def fb_loop():
    mzs = []
    for i in range(N_BC):
        valid = s2[i][s2[i] != 0]
        mzs.append(tok.calculate_precursor_ions(
            valid.unsqueeze(0).cpu(), c2[i:i+1].cpu()).squeeze().item())
    return torch.tensor(mzs, device=DEVICE, dtype=torch.float64)

def fb_vec():
    pad = torch.where(s2 == 0, torch.zeros_like(s2), s2)
    mzs = tok.calculate_precursor_ions(pad.cpu(), c2.cpu()).to(DEVICE, dtype=torch.float64)
    return mzs.squeeze(-1) if mzs.ndim > 1 else mzs

assert fb_loop().shape == fb_vec().shape, 'Shape mismatch'
print('  Shape check: PASSED')
m2l, s2l = bench(fb_loop, n_reps=100, n_warmup=10)
m2v, s2v = bench(fb_vec,  n_reps=100, n_warmup=10)
_sp2 = m2l / m2v
print(f'  Loop ({N_BC} tokenizer calls):  {m2l:.3f} +/- {s2l:.3f} ms')
print(f'  Vectorized (1 batched call): {m2v:.3f} +/- {s2v:.3f} ms')
print(f'  Speedup: {_sp2:.1f}x  (called at every decode step for finished beams, ~12x per spectrum)')
micro_results['fb'] = {'loop': m2l, 'vec': m2v, 'speedup': _sp2}
micro_rows.append({'Optimization': 'P2: _finish_beams',
                   'Before (ms)': round(m2l, 3), 'After (ms)': round(m2v, 3),
                   'Speedup_val': round(_sp2, 1), 'Speedup': f'{_sp2:.1f}x',
                   'Note': 'per-beam loop -> single batched call; measured in isolation'})

# ── MICRO 3 — Patch 5: torch.compile ─────────────────────────────
print('\n' + '='*60)
print('MICRO 3: torch.compile — isolated encoder & decoder latency')
print('='*60)
n_rp  = 123
mzs_s = torch.zeros(1, N_PEAKS, device=DEVICE)
its_s = torch.zeros(1, N_PEAKS, device=DEVICE)
mzs_s[0, :n_rp] = torch.rand(n_rp, device=DEVICE) * 1303 + 301
its_s[0, :n_rp] = torch.rand(n_rp, device=DEVICE)
its_s[0, :n_rp] /= its_s[0, :n_rp].norm()
prc_s = torch.tensor([[(600. - 1.007276) * 2., 2., 600.]], device=DEVICE)
sem   = torch.zeros(1, 0, dtype=torch.int64, device=DEVICE)

enc_b = model_base.encoder; dec_b = model_base.decoder
with torch.no_grad(): mems_b, masks_b = enc_b(mzs_s, its_s)

def _rb():
    with torch.no_grad(): enc_b(mzs_s, its_s)

def _db():
    with torch.no_grad():
        dec_b(tokens=sem, memory=mems_b, memory_key_padding_mask=masks_b, precursors=prc_s)

m4eb, _ = bench(_rb, n_reps=100, n_warmup=20)
m4db, _ = bench(_db, n_reps=100, n_warmup=20)
print(f'  Baseline encoder:     {m4eb:.3f} ms')
print(f'  Baseline decoder x1:  {m4db:.3f} ms')
micro_results['enc'] = {'baseline': m4eb}
micro_results['dec'] = {'baseline': m4db}

if ENABLE_COMPILE:
    enc_o = model_opt.encoder; dec_o = model_opt.decoder
    print(f'  Warming compiled encoder  ({N_COMPILE_WARMUP} iters)...', end=' ', flush=True)
    with torch.no_grad():
        for _ in range(N_COMPILE_WARMUP): enc_o(mzs_s, its_s)
    if DEVICE == 'cuda': torch.cuda.synchronize()
    print('done')
    with torch.no_grad(): mems_o, masks_o = enc_o(mzs_s, its_s)
    print(f'  Warming compiled decoder  ({N_COMPILE_WARMUP} iters)...', end=' ', flush=True)
    with torch.no_grad():
        for _ in range(N_COMPILE_WARMUP):
            dec_o(tokens=sem, memory=mems_o,
                  memory_key_padding_mask=masks_o, precursors=prc_s)
    if DEVICE == 'cuda': torch.cuda.synchronize()
    print('done')

    def _ro():
        with torch.no_grad(): enc_o(mzs_s, its_s)

    def _do():
        with torch.no_grad():
            dec_o(tokens=sem, memory=mems_o,
                  memory_key_padding_mask=masks_o, precursors=prc_s)

    m4eo, _ = bench(_ro, n_reps=100, n_warmup=10)
    m4do, _ = bench(_do, n_reps=100, n_warmup=10)
    _sp5e = m4eb / m4eo; _sp5d = m4db / m4do
    print(f'  Compiled  encoder:    {m4eo:.3f} ms  ({_sp5e:.2f}x)')
    print(f'  Compiled  decoder x1: {m4do:.3f} ms  ({_sp5d:.2f}x)')
    micro_results['enc'].update({'compiled': m4eo, 'speedup': _sp5e})
    micro_results['dec'].update({'compiled': m4do, 'speedup': _sp5d})
    micro_rows.append({'Optimization': 'P5: compile encoder',
                       'Before (ms)': round(m4eb, 3), 'After (ms)': round(m4eo, 3),
                       'Speedup_val': round(_sp5e, 2), 'Speedup': f'{_sp5e:.2f}x',
                       'Note': 'isolated forward pass, synthetic input'})
    micro_rows.append({'Optimization': 'P5: compile decoder x1',
                       'Before (ms)': round(m4db, 3), 'After (ms)': round(m4do, 3),
                       'Speedup_val': round(_sp5d, 2), 'Speedup': f'{_sp5d:.2f}x',
                       'Note': 'isolated single decoder step, synthetic input'})
else:
    print('  ENABLE_COMPILE=False — skipped')

print('\nMICRO-BENCHMARK SUMMARY (batch-size independent, isolated ops)')
print('='*60)
df_micro = pd.DataFrame(micro_rows)
print(df_micro[['Optimization', 'Before (ms)', 'After (ms)', 'Speedup', 'Note']].to_string(index=False))
df_micro.to_csv('results/micro_benchmark_summary.csv', index=False)

MICRO 1: cumulative_masses (nested loop vs tensor gather)
  Equivalence: PASSED
  Loop:        0.7161 +/- 0.0456 ms
  Vectorized:  0.0279 +/- 0.0029 ms
  Speedup:     25.7x  (eliminates batch*beam Python round-trips per decode init)

MICRO 2: _finish_beams (per-beam loop vs batched tokenizer call)
  Shape check: PASSED
  Loop (16 tokenizer calls):  3.565 +/- 0.496 ms
  Vectorized (1 batched call): 0.244 +/- 0.012 ms
  Speedup: 14.6x  (called at every decode step for finished beams, ~12x per spectrum)

MICRO 3: torch.compile — isolated encoder & decoder latency
  Baseline encoder:     8.605 ms
  Baseline decoder x1:  12.586 ms
  Warming compiled encoder  (8 iters)... done
  Warming compiled decoder  (8 iters)... done
  Compiled  encoder:    8.884 ms  (0.97x)
  Compiled  decoder x1: 14.593 ms  (0.86x)

MICRO-BENCHMARK SUMMARY (batch-size independent, isolated ops)
          Optimization  Before (ms)  After (ms) Speedup                                                        Note
 P1: cumu

In [9]:
# ═══════════════════════════════════════════════════════════════════
# CELL 9 — Scaling plots + final summary
# 4 panels: throughput, latency, GPU util, VRAM — all vs batch size
# ═══════════════════════════════════════════════════════════════════
COLORS = {'base': '#D85A30', 'vec': '#378ADD', 'compile': '#9B59B6', 'opt': '#1D9E75'}
MARKERS= {'base': 'o', 'vec': 's', 'compile': '^', 'opt': 'D'}

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle(f'Casanovo: Batch-Size Scaling — {GPU_NAME}', fontweight='bold')

panel_cfg = [
    (axes[0], 'tp',   'Throughput (spec/s)',  'Throughput vs Batch Size'),
    (axes[1], 'lat',  'Mean latency (ms/spec)','Latency vs Batch Size'),
    (axes[2], 'gpu',  'GPU utilization (%)',   'GPU Utilization vs Batch Size'),
    (axes[3], 'vram', 'Peak VRAM (GB)',         'VRAM Usage vs Batch Size'),
]

for ax, metric, ylabel, title in panel_cfg:
    for key, _ in VARIANT_KEYS:
        pd_ = plot_data[key]
        if pd_['bs']:
            ax.plot(pd_['bs'], pd_[metric],
                    color=COLORS[key], marker=MARKERS[key],
                    label=pd_['label'], linewidth=2, markersize=6)
    ax.set_xscale('log', base=2)
    ax.set_xticks(BATCH_SIZES)
    ax.set_xticklabels([str(b) for b in BATCH_SIZES], fontsize=8)
    ax.set_xlabel('Batch size')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=6, frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
    if metric == 'vram':
        ax.axhline(TOTAL_VRAM, color='red', lw=1, ls='--', alpha=0.5, label='Total VRAM')

plt.tight_layout()
plt.savefig('results/batch_scaling_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/batch_scaling_plots.png')

# ── Compute summary statistics for the 6 questions ───────────────
def _best(metric, maximize=True):
    """Return (bs, variant_label, value) for best metric across all valid results."""
    best_val, best_bs, best_lbl = (0. if maximize else float('inf')), None, ''
    for bs in BATCH_SIZES:
        for key, label in VARIANT_KEYS:
            r = sweep_results.get(bs, {}).get(key, {})
            if r.get('oom') or r.get(metric) is None: continue
            v = r[metric]
            if (maximize and v > best_val) or (not maximize and v < best_val):
                best_val, best_bs, best_lbl = v, bs, label
    return best_bs, best_lbl, best_val

_bs_max_tp, _lbl_max_tp, _val_max_tp = _best('throughput', maximize=True)

# Throughput scaling factor from bs=1 to largest valid bs
def _scaling_factor(key):
    tp_bs1  = sweep_results.get(1,  {}).get(key, {}).get('throughput')
    best_tp = max((sweep_results.get(bs, {}).get(key, {}).get('throughput') or 0
                   for bs in BATCH_SIZES), default=None)
    if tp_bs1 and best_tp: return round(best_tp / tp_bs1, 2)
    return 'N/A'

# Best throughput/VRAM tradeoff (highest throughput per GB of VRAM)
_best_ratio, _ratio_bs, _ratio_lbl = 0., None, ''
for bs in BATCH_SIZES:
    for key, label in VARIANT_KEYS:
        r = sweep_results.get(bs, {}).get(key, {})
        if r.get('oom') or not r.get('throughput') or not r.get('vram_gb'): continue
        ratio = r['throughput'] / max(r['vram_gb'], 0.1)
        if ratio > _best_ratio:
            _best_ratio, _ratio_bs, _ratio_lbl = ratio, bs, label

# Speedup of All-patches vs Baseline across batch sizes (does it grow?)
_speedups_by_bs = {}
for bs in BATCH_SIZES:
    b_tp = sweep_results.get(bs, {}).get('base', {}).get('throughput')
    o_tp = sweep_results.get(bs, {}).get('opt',  {}).get('throughput')
    if b_tp and o_tp: _speedups_by_bs[bs] = round(o_tp / b_tp, 3)

_summary_text = f"""
CASANOVO BATCH-SCALING BENCHMARK — FINAL SUMMARY
Generated  : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}
Hardware   : {GPU_NAME} | {TOTAL_VRAM:.1f} GB VRAM
Dataset    : {N_SUBSET_SWEEP} spectra (cycled) | N_TIMING_SWEEP = {N_TIMING_SWEEP}
Variants   : Baseline, Vec P1-4, Compile P5, All P1-5

PERFORMANCE SEPARATION
  Real-time (bs=1, already analyzed in prior report):
    Baseline mean latency: {sweep_results.get(1,{}).get('base',{}).get('mean_ms','N/A')} ms/spec
    All-patches latency  : {sweep_results.get(1,{}).get('opt', {}).get('mean_ms','N/A')} ms/spec
    Speedup at bs=1      : {_speedups_by_bs.get(1, 'N/A')}x

  High-throughput (bs > 1):
"""

for bs in BATCH_SIZES:
    if bs == 1: continue
    r = sweep_results.get(bs, {}).get('opt', {})
    if r.get('oom') or not r.get('throughput'):
        _summary_text += f'    bs={bs}: OOM\n'
    else:
        _summary_text += (f'    bs={bs:<5}: {r["throughput"]:.1f} spec/s | '
                          f'{r["mean_ms"]:.1f} ms/spec | '
                          f'GPU {r["gpu_pct"]:.0f}% | VRAM {r["vram_gb"]:.2f} GB\n')

_summary_text += f"""
SCALING ANALYSIS ANSWERS

Q1. How does throughput scale with batch size?
    Baseline scaling factor (bs=1 -> max valid bs): {_scaling_factor('base')}x
    All-patches scaling factor                     : {_scaling_factor('opt')}x
    Throughput increases with batch size as the GPU becomes more
    saturated. The AR decoder processes all spectra in a batch in
    parallel at each step, so larger batches amortize fixed overhead.

Q2. How does per-spectrum latency change with batch size?
    Latency decreases as batch size grows because the fixed cost
    (encoder forward, Python overhead, memory transfers) is shared
    across more spectra per batch.
    bs=1 baseline : {sweep_results.get(1,{}).get('base',{}).get('mean_ms','N/A')} ms/spec
    Best latency  : {_best('mean_ms', maximize=False)[2]} ms/spec
                    (bs={_best('mean_ms', maximize=False)[0]}, {_best('mean_ms', maximize=False)[1]})

Q3. Does optimization benefit increase, decrease, or remain constant
    as batch size grows?
    Speedup of All-patches vs Baseline by batch size:
"""
for bs, sp in _speedups_by_bs.items():
    _summary_text += f'    bs={bs:<5}: {sp:.3f}x\n'

_summary_text += f"""    Vectorization (P1-P2) reduces Python overhead that is largely
    amortized at large batch sizes, so its benefit may shrink.
    torch.compile (P5) may benefit more at large bs since there is
    more compute to amortize the compiled graph overhead against.

Q4. Which batch size gives the highest throughput?
    Best observed: bs={_bs_max_tp}, {_lbl_max_tp} at {_val_max_tp:.1f} spec/s

Q5. Which batch size gives the best throughput-per-VRAM tradeoff?
    Best throughput/VRAM: bs={_ratio_bs}, {_ratio_lbl}
    ({_best_ratio:.1f} spec/s per GB)

Q6. Are vectorization and compile optimizations more valuable at
    larger batch sizes than at bs=1?
    See speedup table above. Vectorization patches (P1-P2) address
    Python-level loop overhead; this overhead is a smaller fraction
    of total time at large bs (GPU compute dominates), so their
    relative contribution decreases. torch.compile can yield larger
    absolute gains at higher bs where GPU utilization is higher.

ARTIFACTS
  results/batch_scaling_plots.png          <- 4-panel scaling figure
  results/batch_scaling_comparison.csv     <- full metrics table
  results/batch_speedup_table.csv          <- speedup vs baseline
  results/micro_benchmark_summary.csv      <- patch isolation benchmarks
"""

print(_summary_text)
with open('results/batch_scaling_summary.txt', 'w') as fh:
    fh.write(_summary_text)

# ── Optimization summary table ────────────────────────────────────
_opt_summary = pd.DataFrame([
    {'Batch Size': bs,
     'Baseline spec/s': sweep_results.get(bs, {}).get('base', {}).get('throughput', 'OOM'),
     'Vec P1-4 spec/s':    sweep_results.get(bs, {}).get('vec', {}).get('throughput', 'OOM'),
     'Compile P5 spec/s':  sweep_results.get(bs, {}).get('compile', {}).get('throughput', 'OOM'),
     'All P1-5 spec/s':    sweep_results.get(bs, {}).get('opt', {}).get('throughput', 'OOM'),
     'All P1-5 speedup':   _speedups_by_bs.get(bs, 'OOM')}
    for bs in BATCH_SIZES
])
print('\nOPTIMIZATION SPEEDUP BY BATCH SIZE')
print('='*70)
print(_opt_summary.to_string(index=False))
_opt_summary.to_csv('results/optimization_by_bs.csv', index=False)

print('\n-- results/ --')
for f in sorted(os.listdir('results')):
    fp = os.path.join('results', f)
    print(f'  {f:<50} {os.path.getsize(fp)/1024:.1f} KB')

Saved: results/batch_scaling_plots.png

CASANOVO BATCH-SCALING BENCHMARK — FINAL SUMMARY
Generated  : 2026-06-05 17:49
Hardware   : NVIDIA L4 | 23.6 GB VRAM
Dataset    : 5000 spectra (cycled) | N_TIMING_SWEEP = 5000
Variants   : Baseline, Vec P1-4, Compile P5, All P1-5

PERFORMANCE SEPARATION
  Real-time (bs=1, already analyzed in prior report):
    Baseline mean latency: 338.13 ms/spec
    All-patches latency  : 340.68 ms/spec
    Speedup at bs=1      : 0.993x

  High-throughput (bs > 1):
    bs=8    : 14.7 spec/s | 68.3 ms/spec | GPU 17% | VRAM 0.99 GB
    bs=32   : 46.4 spec/s | 21.5 ms/spec | GPU 34% | VRAM 1.09 GB
    bs=128  : 91.2 spec/s | 11.0 ms/spec | GPU 68% | VRAM 1.34 GB
    bs=512  : 110.7 spec/s | 9.0 ms/spec | GPU 85% | VRAM 2.70 GB

SCALING ANALYSIS ANSWERS

Q1. How does throughput scale with batch size?
    Baseline scaling factor (bs=1 -> max valid bs): 32.95x
    All-patches scaling factor                     : 37.66x
    Throughput increases with batch size as the 